# DREAM AI — combined chat + notebook GPU connector

Stop any currently running notebook-only connector cell first, then run this connector. It starts the existing Qwen chat inference worker and the first-party DREAM NOTEBOOK execution worker in the same Kaggle GPU session. It does not start training. Use this to restore DREAM AI chat while keeping the notebook available.

Required Kaggle Secrets: `DREAM_HOST_WORKER_TOKEN`, `HF_TOKEN`, `DREAM_HOST_NOTEBOOK_WORKER_TOKEN`, and `DREAM_NOTEBOOK_ARTIFACT_TOKEN`. These are separate credentials; do not paste them into notebook cells or chat.

The Qwen chat model occupies GPU memory. Do not start native DREAM training or adapter training while chat inference is active. For a training-only session, stop this connector and use the notebook-only GPU connector instead.


In [ ]:
import os
import sys
import time
import hashlib
import subprocess
from pathlib import Path
import requests
import torch
from kaggle_secrets import UserSecretsClient

HOST = "https://dream-host-live.onrender.com"
RAW = "https://raw.githubusercontent.com/mjnr0477/dream-ai/dream-host-runtime/dream-host/"
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before starting DREAM AI.")

secrets = UserSecretsClient()
def read_secret(name):
    try:
        value = str(secrets.get_secret(name) or "").strip()
    except Exception as exc:
        raise RuntimeError("Missing Kaggle Secret: " + name + ". Add it under Add-ons → Secrets.") from exc
    if len(value) < 32:
        raise RuntimeError("Kaggle Secret " + name + " is missing or too short.")
    return value

chat_token = read_secret("DREAM_HOST_WORKER_TOKEN")
hf_token = read_secret("HF_TOKEN")
notebook_token = read_secret("DREAM_HOST_NOTEBOOK_WORKER_TOKEN")
artifact_token = read_secret("DREAM_NOTEBOOK_ARTIFACT_TOKEN")
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("DREAM HOST:", HOST)

chat_source = requests.get(RAW + "worker.py", timeout=60)
chat_source.raise_for_status()
notebook_source = requests.get(RAW + "notebook_worker.py", timeout=60)
notebook_source.raise_for_status()
chat_file = Path("/kaggle/working/dream_chat_worker.py")
notebook_file = Path("/kaggle/working/dream_notebook_worker.py")
chat_file.write_bytes(chat_source.content)
notebook_file.write_bytes(notebook_source.content)
print("Chat worker SHA-256:", hashlib.sha256(chat_source.content).hexdigest())
print("Notebook worker SHA-256:", hashlib.sha256(notebook_source.content).hexdigest())

chat_env = os.environ.copy()
chat_env.update({"DREAM_HOST_URL": HOST, "DREAM_HOST_WORKER_TOKEN": chat_token, "HF_TOKEN": hf_token})
for key in ("DREAM_HOST_NOTEBOOK_WORKER_TOKEN", "DREAM_NOTEBOOK_ARTIFACT_TOKEN", "DREAM_NOTEBOOK_HF_TOKEN", "DREAM_HOST_NOTEBOOK_TRUSTED_MODE"):
    chat_env.pop(key, None)

notebook_env = os.environ.copy()
notebook_env.update({
    "DREAM_HOST_URL": HOST,
    "DREAM_HOST_NOTEBOOK_WORKER_TOKEN": notebook_token,
    "DREAM_NOTEBOOK_ARTIFACT_TOKEN": artifact_token,
    "DREAM_NOTEBOOK_HF_TOKEN": hf_token,
    "DREAM_HOST_NOTEBOOK_TRUSTED_MODE": "1"
})
for key in ("DREAM_HOST_WORKER_TOKEN", "HF_TOKEN"):
    notebook_env.pop(key, None)

print("Starting Qwen chat inference worker and DREAM NOTEBOOK worker…")
chat_process = subprocess.Popen([sys.executable, "-u", str(chat_file)], env=chat_env)
notebook_process = subprocess.Popen([sys.executable, "-u", str(notebook_file)], env=notebook_env)
print("Both worker processes launched. Keep this cell running.")
try:
    while True:
        chat_code = chat_process.poll() if chat_process is not None else None
        notebook_code = notebook_process.poll() if notebook_process is not None else None
        if chat_process is not None and chat_code is not None:
            print("DREAM AI chat worker exited with code", chat_code, "— see its output above. Notebook worker remains active.")
            chat_process = None
        if notebook_process is not None and notebook_code is not None:
            print("DREAM NOTEBOOK worker exited with code", notebook_code, "— see its output above. Chat worker remains active.")
            notebook_process = None
        if chat_process is None and notebook_process is None:
            break
        time.sleep(2)
except KeyboardInterrupt:
    print("Stopping DREAM AI and notebook workers…")
finally:
    for process in (notebook_process, chat_process):
        if process is not None and process.poll() is None:
            process.terminate()
    for process in (notebook_process, chat_process):
        if process is not None:
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
print("DREAM GPU connector stopped.")
